# Capstone: An FP&A Analyst Agent

## What you'll learn
Nothing new. This pulls the whole stage together into one working agent. Work each step
yourself, then compare with the worked solution at the end.

## The brief

Build an agent that drafts the **month-end variance commentary** for the finance team, for
any period they ask about. Using `data/fpa_actuals_budget.csv`:

1. Load the data and add Gross Profit and Operating Income (actual and budget).
2. Connect to your provider (Claude or ChatGPT) with the key kept out of the notebook.
3. Write read-only tools that return actual vs budget summaries and variance splits.
4. Write a system prompt that fixes units, sign conventions, and the commentary format:
   a headline, the top three drivers with numbers, and one recommended next step.
5. Run the agent loop with a step limit and an audit log.
6. Check every number in the commentary against the tool outputs and flag anything unsupported.
7. Produce commentary for Q2 2025 (`2025-04` to `2025-06`) and for Q4 2024
   (`2024-10` to `2024-12`), and say which parts you would still verify by hand.

## Step 1: load and prepare the data

In [ ]:
# Your code

## Step 2: connect to your provider

In [ ]:
# Your code

## Step 3: tools

In [ ]:
# Your code

## Step 4: system prompt

In [ ]:
# Your code

## Step 5: the agent loop with a step limit and audit log

In [ ]:
# Your code

## Step 6: the number check

In [ ]:
# Your code

## Step 7: run it for Q2 2025 and Q4 2024

In [ ]:
# Your code

## Your review

_Replace this text: which drivers did the agent find, did its numbers pass the check, and
what would you verify by hand before sending the commentary?_

## Worked solution

One complete version. Yours may differ and still be correct.

In [ ]:
import json
import pandas as pd

df = pd.read_csv("data/fpa_actuals_budget.csv")

# Derived lines every FP&A pack uses (all amounts in USD thousands).
df["Gross_Profit"] = df["Revenue"] - df["COGS"]
df["Budget_Gross_Profit"] = df["Budget_Revenue"] - df["Budget_COGS"]
df["Operating_Income"] = df["Gross_Profit"] - df["OpEx"]
df["Budget_Operating_Income"] = df["Budget_Gross_Profit"] - df["Budget_OpEx"]




import os
import importlib
from getpass import getpass

# Choose your provider: "anthropic" (Claude) or "openai" (ChatGPT).
PROVIDER = "anthropic"

MODELS = {"anthropic": "claude-opus-5", "openai": "gpt-5.6"}
KEY_NAMES = {"anthropic": "ANTHROPIC_API_KEY", "openai": "OPENAI_API_KEY"}
MODEL = MODELS[PROVIDER]
sdk = importlib.import_module(PROVIDER)   # the anthropic or openai package


def make_client():
    """Create an API client, or return None (with a message) if there is no key."""
    key_name = KEY_NAMES[PROVIDER]
    if not os.environ.get(key_name):
        print(f"{key_name} is not set. Paste your key below, or press Enter to skip.")
        print("(The key stays in memory only. See 02_api_key_and_first_call.ipynb.)")
        try:
            key = getpass(f"{key_name}: ").strip()
        except Exception:
            key = ""
        if not key:
            print("No key, so the model cells in this notebook will be skipped.")
            return None
        os.environ[key_name] = key
    if PROVIDER == "anthropic":
        return sdk.Anthropic()
    return sdk.OpenAI()


client = make_client()
print("Provider:", PROVIDER, "| Model:", MODEL, "| Ready:", client is not None)



def explain_error(error):
    """Turn an API error into a plain message."""
    if isinstance(error, sdk.AuthenticationError):
        return "The API key was rejected. Check that it is correct and still active."
    if isinstance(error, sdk.RateLimitError):
        return "Rate limit or usage quota reached. Wait a minute, or check your account billing."
    if isinstance(error, sdk.APIConnectionError):
        return "Could not reach the API. Check your internet connection or company proxy."
    if isinstance(error, sdk.APIStatusError):
        return f"The API returned an error ({error.status_code}): {error.message}"
    return f"Unexpected API error: {error}"



METRICS = ["Revenue", "COGS", "Gross_Profit", "OpEx", "Operating_Income"]
PRODUCT_LINES = sorted(df["ProductLine"].unique())
REGIONS = sorted(df["Region"].unique())
FIRST_MONTH, LAST_MONTH = df["Month"].min(), df["Month"].max()


def select_rows(month_from, month_to, product_line=None, region=None):
    """Filter the data, raising ValueError with a clear message for bad inputs."""
    for month in (month_from, month_to):
        if len(month) != 7 or not FIRST_MONTH <= month <= LAST_MONTH:
            raise ValueError(f"month '{month}' must be YYYY-MM between {FIRST_MONTH} and {LAST_MONTH}")
    if month_from > month_to:
        raise ValueError("month_from must be on or before month_to")
    if product_line is not None and product_line not in PRODUCT_LINES:
        raise ValueError(f"unknown product_line '{product_line}'. Choose from {PRODUCT_LINES}")
    if region is not None and region not in REGIONS:
        raise ValueError(f"unknown region '{region}'. Choose from {REGIONS}")

    rows = df[(df["Month"] >= month_from) & (df["Month"] <= month_to)]
    if product_line is not None:
        rows = rows[rows["ProductLine"] == product_line]
    if region is not None:
        rows = rows[rows["Region"] == region]
    return rows


def compare(actual, budget):
    """Actual vs budget with the variance in money and percent."""
    return {
        "actual": round(float(actual), 1),
        "budget": round(float(budget), 1),
        "variance": round(float(actual - budget), 1),
        "variance_pct": round(float((actual - budget) / budget * 100), 1),
    }


def get_pnl_summary(month_from, month_to, product_line=None, region=None):
    """Actual vs budget for every P&L line, plus gross margin percent."""
    rows = select_rows(month_from, month_to, product_line, region)
    summary = {m: compare(rows[m].sum(), rows[f"Budget_{m}"].sum()) for m in METRICS}
    summary["Gross_Margin_pct"] = {
        "actual": round(float(rows["Gross_Profit"].sum() / rows["Revenue"].sum() * 100), 1),
        "budget": round(float(rows["Budget_Gross_Profit"].sum() / rows["Budget_Revenue"].sum() * 100), 1),
    }
    return summary


def get_variance_by(metric, month_from, month_to, by):
    """Variance of one metric split by ProductLine or Region, biggest variance first."""
    if metric not in METRICS:
        raise ValueError(f"unknown metric '{metric}'. Choose from {METRICS}")
    if by not in ("ProductLine", "Region"):
        raise ValueError("by must be 'ProductLine' or 'Region'")
    rows = select_rows(month_from, month_to)
    grouped = rows.groupby(by)[[metric, f"Budget_{metric}"]].sum()
    result = {name: compare(r[metric], r[f"Budget_{metric}"]) for name, r in grouped.iterrows()}
    return dict(sorted(result.items(), key=lambda item: -abs(item[1]["variance"])))


def get_monthly_trend(metric, month_from, month_to, product_line=None, region=None):
    """Month-by-month actual vs budget for one metric, with the average FX rate."""
    if metric not in METRICS:
        raise ValueError(f"unknown metric '{metric}'. Choose from {METRICS}")
    rows = select_rows(month_from, month_to, product_line, region)
    grouped = rows.groupby("Month").agg(
        actual=(metric, "sum"), budget=(f"Budget_{metric}", "sum"), fx_rate=("FX_Rate", "mean"))
    return {
        month: {**compare(r["actual"], r["budget"]), "fx_rate": round(float(r["fx_rate"]), 3)}
        for month, r in grouped.iterrows()
    }






MONTH = {"type": "string", "description": "Month in YYYY-MM format"}
METRIC = {"type": "string", "enum": METRICS, "description": "P&L line"}
PRODUCT = {"type": "string", "enum": PRODUCT_LINES, "description": "Optional product line filter"}
REGION = {"type": "string", "enum": REGIONS, "description": "Optional region filter"}

TOOLS = [
    {
        "name": "get_pnl_summary",
        "description": "Actual vs budget for every P&L line (USD thousands) over a month range, "
                       "optionally for one product line and/or region. Includes gross margin %.",
        "parameters": {
            "type": "object",
            "properties": {"month_from": MONTH, "month_to": MONTH,
                           "product_line": PRODUCT, "region": REGION},
            "required": ["month_from", "month_to"],
            "additionalProperties": False,
        },
    },
    {
        "name": "get_variance_by",
        "description": "Variance to budget for one P&L line, split by ProductLine or Region, "
                       "biggest variance first. Use it to find what drove a variance.",
        "parameters": {
            "type": "object",
            "properties": {"metric": METRIC, "month_from": MONTH, "month_to": MONTH,
                           "by": {"type": "string", "enum": ["ProductLine", "Region"]}},
            "required": ["metric", "month_from", "month_to", "by"],
            "additionalProperties": False,
        },
    },
    {
        "name": "get_monthly_trend",
        "description": "Month-by-month actual vs budget for one P&L line, with the average FX "
                       "rate (1.0 = budget rate; below 1.0 means a weaker local currency).",
        "parameters": {
            "type": "object",
            "properties": {"metric": METRIC, "month_from": MONTH, "month_to": MONTH,
                           "product_line": PRODUCT, "region": REGION},
            "required": ["metric", "month_from", "month_to"],
            "additionalProperties": False,
        },
    },
]

FUNCTIONS = {
    "get_pnl_summary": get_pnl_summary,
    "get_variance_by": get_variance_by,
    "get_monthly_trend": get_monthly_trend,
}


def run_tool(name, arguments):
    """Run one tool call. Returns (text for the model, is_error)."""
    if name not in FUNCTIONS:
        return f"Error: unknown tool '{name}'", True
    try:
        return json.dumps(FUNCTIONS[name](**arguments)), False
    except (TypeError, ValueError) as error:
        return f"Error: {error}", True


def provider_tools():
    """Convert TOOLS into the format the chosen provider expects."""
    if PROVIDER == "anthropic":
        return [{"name": t["name"], "description": t["description"],
                 "input_schema": t["parameters"]} for t in TOOLS]
    return [{"type": "function", "name": t["name"], "description": t["description"],
             "parameters": t["parameters"]} for t in TOOLS]


SYSTEM_PROMPT = """You are an FP&A analyst drafting month-end variance commentary for a
fictional company. All amounts are in USD thousands.
Always get numbers from the tools; never guess or invent figures.
For cost lines (COGS, OpEx) a positive variance is unfavourable.
Format:
Headline: one sentence on operating income vs budget.
Drivers: the three biggest drivers, each with its variance and the product line or region.
Next step: one recommended action for the finance team."""


import re


def numbers_in(text):
    """Every number in a piece of text, as floats, ignoring dates and list numbering."""
    text = re.sub(r"\b(19|20)\d{2}-\d{2}\b", " ", text)   # months such as 2025-04
    text = re.sub(r"\bQ[1-4]\b", " ", text)                  # quarters such as Q2
    text = re.sub(r"(?m)^\s*\d+[.)]\s", " ", text)          # list numbering such as "1. "
    numbers = [float(n.replace(",", "")) for n in re.findall(r"-?\d[\d,]*\.?\d*", text)]
    return [n for n in numbers if not (n.is_integer() and 1900 <= n <= 2100)]   # years


def check_answer(answer, tool_outputs, tolerance=0.005):
    """Return the numbers in the answer that do not match any number a tool returned.

    Answers often round (1,234.5 written as 1,235), so a number counts as supported when
    it is within `tolerance` (0.005 = 0.5%) of a tool number, or within 0.1 for small values.
    Signs are ignored, because "a miss of 50" and "-50" describe the same variance.
    """
    source = [abs(n) for output in tool_outputs for n in numbers_in(output)]
    unsupported = []
    for n in numbers_in(answer):
        if not any(abs(abs(n) - s) <= max(0.1, tolerance * s) for s in source):
            unsupported.append(n)
    return unsupported


def run_guarded_agent(question, system=SYSTEM_PROMPT, max_steps=8):
    """The agent loop with guardrails: step limit, audit log, and a number check."""
    if client is None:
        return None
    tools = provider_tools()
    log = []   # every tool call: name, inputs, output, error flag

    def call_tool(name, arguments):
        output, is_error = run_tool(name, arguments)
        log.append({"tool": name, "inputs": arguments, "output": output, "error": is_error})
        print(f"step {len(log)}: {name}({arguments}){'  <- error' if is_error else ''}")
        return output, is_error

    answer = None
    try:
        if PROVIDER == "anthropic":
            messages = [{"role": "user", "content": question}]
            for _ in range(max_steps):
                response = client.messages.create(
                    model=MODEL, max_tokens=16000, system=system, tools=tools, messages=messages)
                if response.stop_reason == "refusal":
                    return {"answer": "The model declined this request.", "log": log, "needs_review": True}
                messages.append({"role": "assistant", "content": response.content})
                calls = [b for b in response.content if b.type == "tool_use"]
                if not calls:
                    answer = "".join(b.text for b in response.content if b.type == "text")
                    break
                results = []
                for call in calls:
                    output, is_error = call_tool(call.name, call.input)
                    results.append({"type": "tool_result", "tool_use_id": call.id,
                                    "content": output, "is_error": is_error})
                messages.append({"role": "user", "content": results})
        else:
            items = [{"role": "user", "content": question}]
            for _ in range(max_steps):
                response = client.responses.create(
                    model=MODEL, instructions=system, tools=tools, input=items)
                items += response.output
                calls = [item for item in response.output if item.type == "function_call"]
                if not calls:
                    answer = response.output_text
                    break
                for call in calls:
                    output, _ = call_tool(call.name, json.loads(call.arguments))
                    items.append({"type": "function_call_output", "call_id": call.call_id,
                                  "output": output})
    except sdk.APIError as error:
        print(explain_error(error))
        return None

    if answer is None:
        return {"answer": f"Stopped after {max_steps} steps.", "log": log, "needs_review": True}

    unsupported = check_answer(answer, [entry["output"] for entry in log if not entry["error"]])
    return {
        "answer": answer,
        "log": log,
        "unsupported_numbers": unsupported,
        "needs_review": bool(unsupported) or not log,
    }


for label, start, end in [("Q2 2025", "2025-04", "2025-06"), ("Q4 2024", "2024-10", "2024-12")]:
    print("=" * 70)
    print(label)
    print("=" * 70)
    result = run_guarded_agent(f"Write the variance commentary for {start} to {end}.")
    if result:
        print("\n" + result["answer"])
        print("\nUnsupported numbers:", result.get("unsupported_numbers"))
        print("Needs human review:", result["needs_review"], "\n")

## Where to go next

You have built a working AI agent on business data: tools, a loop, memory, and guardrails.
To keep growing:

- Point the same tools at a real (approved) dataset and compare the agent's commentary with your own.
- Add tools for forecasts or headcount, and describe them well enough that the agent uses them.
- Continue to **Stage 7: Machine Learning**, where you build models that learn from data.